# Pillar 1: Shock Absorption (Rainfall-Based)

Model:
income_it = beta1*mgnrega_it + beta2*rainfall_shock_it + beta3*(mgnrega_it*rainfall_shock_it) + FE + epsilon

Primary metric: beta3

In [1]:
from pathlib import Path
import pandas as pd
from analysis_utils import load_and_preprocess, fit_fe_clustered, minmax_normalize, compute_vif, summarize_model

DATA_PATH = Path('Panel_Data 2014-24.csv')
OUT_DIR = Path('outputs')
OUT_DIR.mkdir(exist_ok=True)

df = load_and_preprocess(str(DATA_PATH))
df['mgnrega_x_shock'] = df['mgnrega'] * df['rainfall_shock']

In [2]:
formula = 'income ~ mgnrega + rainfall_shock + mgnrega:rainfall_shock + C(region_id) + C(year)'
results, used_df = fit_fe_clustered(formula, df)
print(results.summary())

coef_table = summarize_model(results, ['mgnrega', 'rainfall_shock', 'mgnrega:rainfall_shock'])
coef_table.to_csv(OUT_DIR / 'pillar1_coefficients.csv', index=False)
print(coef_table)

vif = compute_vif(used_df, ['mgnrega', 'rainfall_shock', 'mgnrega_x_shock'])
vif.to_csv(OUT_DIR / 'pillar1_vif.csv', index=False)
print(vif)

beta3 = results.params.get('mgnrega:rainfall_shock', 0.0)
region_raw = used_df.groupby('region_id')['mgnrega_x_shock'].mean() * beta3
pillar1_score = minmax_normalize(region_raw).rename('pillar1_shock_score').reset_index()
pillar1_score.to_csv(OUT_DIR / 'pillar1_shock_score.csv', index=False)
pillar1_score.head()

                            OLS Regression Results                            
Dep. Variable:                 income   R-squared:                       0.685
Model:                            OLS   Adj. R-squared:                  0.649
Method:                 Least Squares   F-statistic:                     128.1
Date:                Mon, 13 Apr 2026   Prob (F-statistic):          1.51e-100
Time:                        22:34:03   Log-Likelihood:                -25042.
No. Observations:                2560   AIC:                         5.062e+04
Df Residuals:                    2292   BIC:                         5.219e+04
Df Model:                         267                                         
Covariance Type:              cluster                                         
                                                  coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

C:\Users\Tanishq op\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\statsmodels\base\model.py:1896: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 267, but rank is 12
  warnings.warn('covariance of constraints does not have full '


,region_id,pillar1_shock_score
0,ADILABAD_TELANGANA,0.531386
1,AGRA_UTTAR PRADESH,0.394338
2,AJMER_RAJASTHAN,0.346876
3,ALAPPUZHA_KERALA,0.593730
4,ALIGARH_UTTAR PRADESH,0.514880
